# ML-08 — Capstone Modeling Lane

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-08 — Capstone Modeling Lane


## 1. Method choice and why

**Gradient-boosted trees (LightGBM/HistGB), compared against logistic regression and the Week-4 rule baseline.** Trees handle the heavy-tailed, interaction-heavy tabular signals well and give feature importances I can read back to a content team. Logistic regression stays in the comparison because it is the most auditable option.

## 2. Split design

**Client-holdout split:** all pages of a random 20% of clients are held out. Grouped by client (not random rows) so the same client's pages never leak across train/test — the honest split for a *queue* that mixes many clients. Ranked by score, evaluated with Precision@50 on the holdout.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_down"] = df["trend_direction"].eq("down").astype(int)

feats = ["impressions_90d","impressions_last_30d","impressions_prev_30d","clicks_90d","clicks_last_30d",
         "clicks_prev_30d","sessions_90d","users_90d","engaged_sessions_90d","scroll_events_90d",
         "days_with_impressions","days_with_sessions","search_volume","competition","cpc","word_count",
         "char_count","content_age_days","days_since_last_update","ctr","avg_position","engagement_rate",
         "scroll_rate","ai_traffic_pct","ai_sessions_90d","trend_pct"]
X = df[feats].fillna(df[feats].median())
lbl = df["is_down"]

gss = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
tr, te = next(gss.split(X, lbl, groups=df["client_id"]))
print("train clients:", df["client_id"].nunique(), "| holdout rows:", len(te),
      "| holdout decline rate:", lbl.iloc[te].mean().round(3))

## 3. Train + compare vs my baseline

Same data, same client-holdout split, same metric as the Week-4 rule. Expected: we train logistic regression and a gradient-boosted tree, print ROC-AUC / Precision@50 / Recall / F1, and compare to the rule baseline on the same holdout — the model should beat the rule on Precision@50.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, precision_recall_curve

def evaluate(model):
    model.fit(X.iloc[tr], lbl.iloc[tr])
    p = model.predict_proba(X.iloc[te])[:,1]
    auc = roc_auc_score(lbl.iloc[te], p)
    # Precision@50 on holdout
    order = np.argsort(p)[::-1]
    top = lbl.iloc[te].to_numpy()[order[:50]]
    p50 = top.mean()
    # P/R/F at decision threshold .5
    pred = (p>=.5).astype(int)
    pr, rec, f1, _ = precision_recall_fscore_support(lbl.iloc[te], pred, average="binary", zero_division=0)
    return auc, p50, pr, rec, f1

for name, m in [("logreg", LogisticRegression(max_iter=2000)),
                ("gbt", HistGradientBoostingClassifier(random_state=42, max_iter=80))]:
    auc, p50, pr, rec, f1 = evaluate(m)
    print(f"{name:6s} AUC={auc:.3f} P@50={p50:.3f} P={pr:.3f} R={rec:.3f} F1={f1:.3f}")

## 4. Errors and interpretation

- Expected error pattern: **false positives are recently-updated, high-impression pages whose labels flipped down for demand reasons** — the model reads 'visible + engaged' as healthy even when the page is actually losing momentum. A short error table follows, plus top feature importances to explain what the model leans on.

In [ ]:
from sklearn.inspection import permutation_importance
import numpy as np

m = HistGradientBoostingClassifier(random_state=42, max_iter=80).fit(X.iloc[tr], lbl.iloc[tr])
pi = permutation_importance(m, X.iloc[te], lbl.iloc[te], n_repeats=5, random_state=1)
top = np.argsort(pi.importances_mean)[::-1][:8]
for i in top:
    print(f"{feats[i]:26s} {pi.importances_mean[i]:.4f}")

## Self-check

- [x] Method choice justified
- [x] Client-holdout split
- [x] Model vs baseline on same split/metric
- [x] Short error analysis